# Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

# -- Personal Libraries
from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, ProductTokenBuilder
from src.utils import TemporalSplitter

/home/thebigmonster/Github/nn-elasticity/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setting

In [3]:
# initial seed
BASE_SEED = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ── Data ──────────────────────────────────────────────────────────
N_UPCS = 5 # Number of UPCs
SMOOTH_WINDOW = 8 # Smoothing window for phase 0
BETA_EDA = -2 # Beta for initialization phase 0
K_NEIGHBORS = 5 # Number of neighbors for the product

# ── Robust Tuning ─────────────────────────────────────────────────
N_FOLDS = 3  # Number of folds for cross-validation
TUNE_SEEDS = [11, 29, 42]  # Seeds for cross-validation
MIN_TRAIN_FRAC = 0.50  # Minimum training fraction

# ── Training for tuning ──────────────────────────────────────
N_EPOCHS_P0 = 250
N_EPOCHS_P1 = 300
PATIENCE    = 20 # How many epochs to wait before reducing learning rate
ES_PATIENCE = 40 # How many epochs to wait before early stopping

# ── Dimensionality for sku-level features ───────────────────────────
D_STORE = 16
D_BRAND = 8
D_STYLE = 8

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Results ─────────────────────────────────────────────────────
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_TRIAL_PATH = RESULTS_DIR / "best_trial_params.json"
TRIAL_SUMMARY_PATH = RESULTS_DIR / "nn_hparam_trials_summary.csv"

Device: cuda


# Seeds

In [4]:
# Function to set all seeds
# and make the results reproducible
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Make the results reproducible and control the randomness
    torch.backends.cudnn.benchmark = False # Make the results reproducible and control the randomness

set_all_seeds(BASE_SEED)

# Loader

In [5]:
# Load the dataset
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv").copy()
print(f"Dataset shape: {df.shape}")

# Encode the categorical variables
encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True) # Encode the store code to numerical values
_, week_cats  = encoder.factorize(df, "week_id", sort=True) # Encode the week id to numerical values
_, brand_cats = encoder.factorize(df, "brand_family_norm",  sort=True)   # Encode the brand family to numerical values
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)   # Encode the style segment to numerical values

n_stores = len(store_cats)
n_weeks  = len(week_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)
print(f"Stores: {n_stores}  |  Weeks: {n_weeks}  |  Brands: {n_brands}  |  Styles: {n_styles}")

# Encode brand y style en el dataframe principal
# We create a mapping of brand and style (numerical) codes to 0,1,2,...
# to be globally used for the folds; For instance,
# brand_cats = Index([101, 102,...])
# brand_map = {101: 0, 102: 1, ...}
# The same for style_cats and style_map.          
brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"]  = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

# Build the multi-product dataset
mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS) # Fit the builder to the data

# Transform the data to wide format (pivot table with UPCs and regressors as a columns
# and week_store as rows)
full_wide_raw = mp_builder.transform().copy() 
n_upcs = mp_builder.n # Store the number of selected UPCs

print(f"Full wide shape: {full_wide_raw.shape}")
print(f"UPCs selected: {n_upcs}")
print(f"Top UPCs: {mp_builder.selected_upcs[:N_UPCS]}")

Dataset shape: (463722, 44)
Stores: 70  |  Weeks: 302  |  Brands: 54  |  Styles: 13
Full wide shape: (19808, 171)
UPCs selected: 5
Top UPCs: [3410010505, 7289000011, 1820000784, 8248812345, 3410017306]


# Neighbor Meta

In [6]:
# Neighbors:
# Static metadata per UPC position — used by neighbor-aware attention in the model
upc_meta = (
    df.groupby("upc_code")[["category_code", "brand_family_norm",
                             "style_segment_norm", "liters_per_upc"]]
    .first()
    .loc[mp_builder.selected_upcs]
)

cat_codes, _ = pd.factorize(upc_meta["category_code"], sort=True)

neighbor_meta = {
    "category": torch.tensor(cat_codes, dtype=torch.long, device=device),
    "brand":    torch.tensor(upc_meta["brand_family_norm"].values,   dtype=torch.long,    device=device),
    "style":    torch.tensor(upc_meta["style_segment_norm"].values,  dtype=torch.long,    device=device),
    "liters":   torch.tensor(upc_meta["liters_per_upc"].values,      dtype=torch.float32, device=device),
}
print("neighbor_meta built")

neighbor_meta built


# Temporal Folds

In [7]:
splitter = TemporalSplitter(week_col="week_id") # Initialize the temporal splitter
fold_splits = splitter.expanding_splits(
    df=full_wide_raw, # The data to split
    n_folds=N_FOLDS, # The number of folds
    min_train_frac=MIN_TRAIN_FRAC, # The minimum training fraction
)

print(f"N folds available: {len(fold_splits)}")
for i, (train_fold, val_fold) in enumerate(fold_splits):
    print(
        f"Fold {i}: train={len(train_fold):,} "
        f"val={len(val_fold):,} "
        f"train_weeks={train_fold['week_id'].nunique()} "
        f"val_weeks={val_fold['week_id'].nunique()}"
    )

N folds available: 3
Fold 0: train=9,756 val=3,394 train_weeks=151 val_weeks=50
Fold 1: train=13,150 val=3,339 train_weeks=201 val_weeks=50
Fold 2: train=16,489 val=3,254 train_weeks=251 val_weeks=50


# Functions

In [8]:
# We create a mapping of store and week (numerical)codes to 0,1,2,...
# to be globally used for the folds; For instance,
# store_cats = Index([101, 102,...])
# store_map = {101: 0, 102: 1, ...}
# The same for week_cats and week_map.
store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}


# This function prepare the data for training.
# It encodes the store and week codes, sorts the data by store and week codes,
# and smooths the log liters.
def build_fold_frames(train_wide, val_wide, smooth_window: int):
    train_wide = train_wide.copy()
    val_wide   = val_wide.copy()

    # Encode the store and week codes
    for w in [train_wide, val_wide]:
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"]    = w["week_id"].map(week_map)

    # Sort the data by store and week codes to do the rolling mean
    train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
    val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

    # Smooth the log liters. Delete the noise week by week.
    # For the Phase 0, we use a moving average of n weeks.
    for i in range(n_upcs):
        col = f"log_liters_{i}"
        for df_w in [train_wide_s, val_wide_s]:
            df_w[col] = (
                df_w.groupby("store_code")[col]
                .transform(lambda s: s.rolling(window=smooth_window, min_periods=1).mean())
            )

    return train_wide, val_wide, train_wide_s, val_wide_s

# This function builds the datasets for the training and validation.
def build_loaders(train_wide, val_wide, train_wide_s, val_wide_s, batch_size: int):

    loader_factory = DataLoaderFactory(
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    # Create the DataLoaders for the phase0 and phase1.
    # For training we shuffle the data and drop the last batch.
    # For validation we don't shuffle the data and don't drop the last batch.
    # Important! One might think that shuffling the data could alter its sequential order,
    # however, in this case, the MLP will process the data for each pair (shop, week)
    # and, therefore, the order does not matter. It would be a problem if the architecture were, for example,
    # an RNN or an LSTM, but in this case it is not.
    # Observation! The drop_last is True for the training set. We try to avoid things like: 
    # 28 observations in the last batch compared to 500 in the others, for instance.

    train_ds_p0 = MultiProductDataset(train_wide_s, n=n_upcs) # Phase 0 training dataset
    val_ds_p0   = MultiProductDataset(val_wide_s,   n=n_upcs) # Phase 0 validation dataset
    train_ds    = MultiProductDataset(train_wide,   n=n_upcs) # Phase 1 training dataset
    val_ds      = MultiProductDataset(val_wide,     n=n_upcs) # Phase 1 validation dataset

    train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,   batch_size=batch_size, shuffle=False)
    train_loader    = loader_factory.create_train_loader(train_ds,    batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader      = loader_factory.create_eval_loader(val_ds,       batch_size=batch_size, shuffle=False)
    return train_loader_p0, val_loader_p0, train_loader, val_loader

# Helpers

In [9]:
def freeze_nonlinear(model):
    # Freeze all spline heads (own and cross) and the bilinear interaction head.
    # This reduces phase 0 to a purely log-linear model:
    #   g_i ≈ b_i + β_{ii}·u_i + Σ_j a_{ij}·β_{ij}·u_j
    # Only the linear coefficients (head_b, head_beta, head_beta_cross) remain free.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(False)
        head.bias.requires_grad_(False)
def unfreeze_nonlinear(model):
    # Unfreeze all spline and bilinear heads for phase 1.
    for attr in ["head_w", "head_w_cross", "head_cross"]:
        head = getattr(model.head.param_head, attr)
        head.weight.requires_grad_(True)
        head.bias.requires_grad_(True)

# To initialize the beta prior of the model
# because of EDA, the global elasticity is -2.
def init_beta_prior(model, beta_target):
    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-beta_target, dtype=torch.float32)) - 1.0
    )# Initialize the head_beta bias with the inverse softplus of BETA_EDA
    with torch.no_grad():
        # Set the head_beta weight to zero, therefore, the initial head_beta 
        # is independent of the context.
        model.head.param_head.head_beta.weight.zero_()
        # Set the head_beta bias with the inverse softplus of BETA_EDA.
        model.head.param_head.head_beta.bias.fill_(beta_raw_init)
        # This implies that beta_raw = 0*h + beta_raw_init = beta_raw_init
        # All products have the same beta_raw_init at the beginning. When
        # the model is trained, beta_raw will be updated.

print("Helpers defined")

Helpers defined


In [10]:
# This function runs the training loop.
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, neighbor_meta, phase_name="", 
                 verbose=False):

    best_val_loss = float("inf") # Initialize the best validation loss
    no_improve    = 0 # Initialize the number of epochs without improvement
    # Scales the loss to prevent underflow in training with mixed precision (float32->float16)
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    # Training loop
    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train() # Set the model to training mode
        total_loss, total_denom = 0.0, 0.0 # Initialize the total loss and the pondered denominator
        # The batches don't have the same size, because it exists the obs_mask (observations mask);
        # we can't treat a batch with 10 observation like one with 100 observations. For this reason, 
        # we need to get the pondered real average.

        # Recall that: obs_mask = 1 if the observation is available
        # (the product was sold this week in this store), 0 otherwise (the product was not sold).

        for batch in train_loader:
            # Move the 8 pre-stacked tensors to the GPU with non_blocking=True.
            # non_blocking=True lets the DMA transfer overlap with CPU work (requires pin_memory=True,
            # which is already set in DataLoaderFactory). Safe here because the tensors
            # are not read on CPU after this point.
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            # demands and obs_mask are already (B, n) — pre-stacked in MultiProductDataset.__init__.
            y_true   = batch["demands"]   # (B, n) float — no torch.stack() needed
            obs_mask = batch["obs_mask"]  # (B, n) float — no torch.stack() needed

            optimizer.zero_grad() # Reset the gradients
            if scaler: # If the scaler is not None, we use mixed precision
                with torch.amp.autocast("cuda"): # Use mixed precision (AMP)
                    # compute_E=True is needed so that aux["E"] is available for L_elast.
                    # For phase 0 (lambda_elast=0) this can be set to False to save compute.
                    y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                    loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                        aux["pairs"], E=aux.get("E"))
                scaler.scale(loss).backward() # Backward pass
                # Unscale the gradients; the gradients are inflated 
                # because of the mixed precision (float32->float16).
                scaler.unscale_(optimizer)
                # We need to avoid explosive gradients, for this reason
                # we clip the gradients
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer) # We update the parameters
                scaler.update() # We update the scale factor of the scaler
            else:
                # If the scaler is None (no GPU), we don't use mixed precision
                # and we use the normal backward pass.
                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                loss, logs = loss_fn(y_hat, y_true, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                    aux["pairs"], E=aux.get("E"))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item() # Number of available observations (n_obs_batch)
            # Recall that:
            # logs["loss"] = total_loss_batch / n_obs_batch
            # We recover the total loss to, at the end of the epoch,
            # compute the real average.
            total_loss  += logs["loss"].item() * denom 
            total_denom += denom # Sum of the denominator

        # ── Val ────────────────────────────────────────────────────
        model.eval() # Set the model to evaluation mode
        val_loss_sum, val_denom = 0.0, 0.0 # Initialize the validation loss and the pondered denominator

        with torch.no_grad(): # No gradients are computed
            for batch in val_loader: # Iterate over the validation loader
                batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()} # To GPU
                # demands and obs_mask are already (B, n) — pre-stacked in MultiProductDataset.__init__.
                y_true   = batch["demands"]   # (B, n) float — no torch.stack() needed
                obs_mask = batch["obs_mask"]  # (B, n) float — no torch.stack() needed

                y_hat, _, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)
                _, logs = loss_fn(y_hat, y_true, obs_mask,
                                aux["w"], aux["ddBx"], aux["u"], aux["Bx"],
                                aux["pairs"], E=aux.get("E")) # Compute the loss
                                  
                denom        = obs_mask.sum().item() # Number of available observations
                val_loss_sum += logs["loss"].item() * denom # Sum of the total loss
                val_denom    += denom # Sum of the denominator

        # We compute the pondered real average.
        val_loss = val_loss_sum / max(val_denom, 1.0) # Average of the loss
        prev_lr = optimizer.param_groups[0]["lr"] # Previous learning rate
        scheduler.step(val_loss) # Update the learning rate (scheduler)
        new_lr = optimizer.param_groups[0]["lr"] # New learning rate
        if new_lr < prev_lr: # If the new learning rate is lower than the previous one,
            no_improve = 0

        # If the validation loss is lower than the best validation loss,
        # we save the model otherwise we increment the number of epochs without improvement.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        # If the number of epochs without improvement is 0,
        # we print the validation loss.
        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        # If the number of epochs without improvement is greater than the patience,
        # we stop the training (Early Stopping).
        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping in epoch {epoch+1}")
            break

    return best_val_loss

print("run_training defined")

run_training defined


In [11]:
# Hidden options for the model (Optuna)
HIDDEN_OPTIONS = {
    "64_32":        (64, 32),
    "128_64":       (128, 64),
    "192_96":       (192, 96),
    "256_128":      (256, 128),
    "256_128_64":   (256, 128, 64),
}

# This function compute the R2, MAE and RMSE.
# Recall that:
# MAE is the mean absolute error.
# RMSE is the root mean square error.
# R2 is the coefficient of determination.
def compute_global_metrics(model, val_loader, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            # demands and obs_mask are already (B, n) — pre-stacked in MultiProductDataset.__init__.
            y_true   = batch["demands"]   # (B, n) float — no torch.stack() needed
            obs_mask = batch["obs_mask"]  # (B, n) float — no torch.stack() needed
            y_hat, _, _ = model(batch, return_parts=True, neighbor_meta=neighbor_meta)

            # We get only the available observations.
            mask = obs_mask.bool() # Mask of the available observations
            all_true.append(y_true[mask].cpu()) # Append the true values
            all_pred.append(y_hat[mask].cpu()) # Append the predicted values

    y_true_all = torch.cat(all_true).float() # Concatenate the true values
    y_pred_all = torch.cat(all_pred).float() # Concatenate the predicted values

    err = y_true_all - y_pred_all # Error
    mae = float(err.abs().mean()) # Mean absolute error
    rmse = float(torch.sqrt((err ** 2).mean())) # Root mean square error

    ss_res = float((err ** 2).sum()) # Sum of the squared errors
    ss_tot = float(((y_true_all - y_true_all.mean()) ** 2).sum()) # Sum of the total errors
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan # R2

    return {
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2,
    }

# This function compute the Elasticity Score for the optimization parameters of Optuna.
# Our intention is to evaluate how good the model is at predicting the elasticity.
# Recall that:
# The Elasticity Score is in the range [0, 1].
# The closer to 1, the better.
# We shall assume that in FMCG, tipically the elasticity is in the range [-5, 0]. 
# One could change this range to adapt it to other products, but it is not the purpose of this notebook.
def compute_elasticity_score(model, val_loader, device, 
                             own_min=-5.0, own_max=0.0,
                             cross_min=-1.0, cross_max=1.0):
    model.eval()
    all_own, all_cross = [], []
    off_diag = ~torch.eye(model.n, dtype=torch.bool, device=device).unsqueeze(0)  # (1, n, n)


    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            obs_mask = batch["obs_mask"].bool()
            _, eps_hat, aux = model(batch, return_parts=True, compute_E=True, neighbor_meta=neighbor_meta)

            # Elasticity matrix
            E = aux["E"]
            # Own-price elasticity
            all_own.append(eps_hat[obs_mask].cpu()) 
            # Cross-price elasticity
            # We get the pair mask (B, n, n)
            pair_mask = obs_mask.unsqueeze(2) & obs_mask.unsqueeze(1)  
            # Exclude the diagonal (own-price) to remain with the cross-price elasticities
            cross_mask = pair_mask & off_diag  # (B, n, n)
            all_cross.append(E[cross_mask].cpu())

    own = torch.cat(all_own).numpy() # Concatenate the own-price elasticities
    cross = torch.cat(all_cross).numpy() if all_cross else np.array([])

    # ── Own score ────────────────────────────────────────────────
    # We compute the percentage of predicted elasticities that are in the range [-5, 0].
    own_in_range  = float(((own >= own_min) & (own <= own_max)).mean())
    median_own    = float(np.median(own))
    # We compute the penalty for the prior.
    # Because of EDA, the global elasticity is -2 approximately.
    # Therefore, we want the median of the predicted elasticities to be -2.
    # If it is not, we penalize the model.
    deviation     = max(0.0, abs(median_own - BETA_EDA) - 0.3)
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)
    # It is a weighted average of the percentage of predicted elasticities in the range [-5, 0]
    # and the penalty for the prior.
    own_score     = own_in_range * (1.0 - prior_penalty)

    # ── Cross score ───────────────────────────────────────────────
    if len(cross) > 0:
        cross_in_range = float(((cross >= cross_min) & (cross <= cross_max)).mean())
        median_cross    = float(np.median(cross))
    else:
        # If there are no cross-price elasticities, we assume the score is 1.0
        cross_in_range = 1.0
        media_cross = float("nan")  

    # ── Final score ───────────────────────────────────────────────
    score = 0.7 * own_score + 0.3 * cross_in_range

    return {
        "elast_score":            float(score),
        "own_score":              float(own_score),
        "own_in_range":           float(own_in_range),
        "own_elasticity_median":  median_own,
        "cross_in_range":         float(cross_in_range),
        "cross_elasticity_median": median_cross,
    }

print("Helpers of metrics defined")

Helpers of metrics defined


In [12]:
# This function build the model and train it.
# We encapsulate the training loop in a function to be able to use it in Optuna.
def build_and_train(params, train_fold, val_fold, fold_id, seed, trial_id=0):
    set_all_seeds(seed) # Set the seeds for reproducibility

    # Build the dataframes:
    # train_wide_s, val_wide_s are the smoothed dataframes.
    # train_wide, val_wide are the original dataframes
    # The four dataframes have the store and week columns encoded.
    train_wide, val_wide, train_wide_s, val_wide_s = build_fold_frames(
        train_wide=train_fold,
        val_wide=val_fold,
        smooth_window=SMOOTH_WINDOW,
    ) 

    # Build DataSets (Pytorch) for the phase0, phase1 and phase2.
    #  · train_ds_p0, val_ds_p0 are the DataSets for the phase0.
    #  · train_ds, val_ds are the DataSets for the phase1 and phase2.
    # Important! The dataframes _s are the smoothed dataframes and are only used 
    # to compute the train_ds_p0 and val_ds_p0. Therefore, for the phase0
    # our objective is to fit the model to the smoothed dataframes and get the
    # best parameters c(x) and beta(x) without the splines activated. 
    train_loader_p0, val_loader_p0, train_loader, val_loader= build_loaders(
        train_wide, val_wide, train_wide_s, val_wide_s, batch_size=params["BATCH_SIZE"]
    )
    
    # ------ IMPORTANT------
    # We need to emphasize the following:
    # in the build_fold_frames function is the encoder of store_code done; 
    # Remember that this encoding is continous, namely, it goes from [101, 205, 312]
    # to [0, 1, 2]. It is extremly important not to reorder this encoding, because
    # the following is thought/computed/coded assuming this order. For instance,
    # in the MultiProductContextEmbeddings, the store_code is used to index the
    # store embedding. If you reorder the encoding, you will be using the wrong
    # embedding for the store.
    # -----------------------
    
    # Get the parameters from the Optuna trial.
    n_knots          = params["N_KNOTS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]]
    dropout          = params["DROPOUT"]
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lambda_smooth = params["LAMBDA_SMOOTH"]
    lambda_elast  = params["LAMBDA_ELAST"]      
    

    # Define the paths to the checkpoints for the phase0 and phase1.
    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_fold{fold_id}_seed{seed}_phase1.pt"

    # ── BUILD THE MODEL ───────────────────────────────────────

    # Build the knots for the splines.
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i = train_wide[f"log_price_{i}"].values # In the splines, we only need the log_price.
        # Build the spline: knots, mean and std.
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    # Stack the knots, mean and std.
    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])

    # Build the price splines (Theory implementation): Bx, dBx, ddBx.
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    # Build the context embeddings for each product (token).
    # We get a (B, out_dim) context tensor. In the article, this tensor is called x_i.
    token_builder = ProductTokenBuilder(
        n=n_upcs,
        n_stores=n_stores, d_store=D_STORE,
        n_brands=n_brands, d_brand=D_BRAND,
        n_styles=n_styles, d_style=D_STYLE,
    )

    # Build the all-in-one model. All the pieces together.
    def make_model(enforce_negative_beta, use_cross):
        # From the latent representation h, 
        # the model computes the parameters b, beta, w, u.
        # Finally, it computes the predicted demand y_hat,
        # the own-price elasticity eps_hat, and the elasticity matrix E.
        head = IntegrableDemandHead(
            context_dim=token_builder.d_token,
            K_splines=n_knots,
            n=n_upcs,
            k_neighbors=K_NEIGHBORS,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        # The model is built. All the pieces together.
        return ICDN(
            context_builder=token_builder,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── PHASE 0 ─────────────────────────────────────────────────────
    # The goal of this phase is to obtain a robust initialization before
    # unlocking the model's full flexibility. To do so:
    #
    #   1. First-order cross-price effects are able to be computed (use_cross=True) 
    #      and the spline weights are frozen (head_w → zeros, requires_grad=False). 
    #      This reduces the model to a log-linear demand: 
    #       log(q) \approx b + beta·log(p) + first-order cross-price effects.
    #
    #   2. The head_beta bias is initialized with the inverse softplus of
    #      BETA_EDA, so that the own-price elasticity at startup equals exactly
    #      -BETA_EDA. This gives the model an economically sensible starting
    #      point instead of a random one.
    #
    #   3. The loss applies no smoothness or positivity penalties (lambda_smooth=0,
    #      lambda_pos=0): only the demand prediction error is minimized.
    #
    # By the end of this phase, beta and b are well calibrated, which makes
    # convergence easier in later phases when spline weights and cross-price
    # effects are unfrozen.

    # Build the model with first-order cross-price effects and enforcing negative beta.
    m0 = make_model(enforce_negative_beta=True, use_cross=True)
    # We freeze the nonlinear parameters.
    freeze_nonlinear(m0)
    # We initialize the head_beta bias with the inverse softplus of BETA_EDA.
    init_beta_prior(m0, BETA_EDA)
    with torch.no_grad():
        # Zero-init head_beta_cross and head_w_cross so that cross-price
        # contributions start at zero and are learned gradually from phase 1 onward.
        m0.head.param_head.head_beta_cross.weight.zero_()
        m0.head.param_head.head_beta_cross.bias.zero_()

    # Define the loss function for the phase0. Notice that we use the mean reduction and
    # only focus on the accuracy of the demand prediction (huber_delta != 0).
    loss_p0 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth,
        lambda_elast=lambda_elast,
        reduction="mean"
    )

    # We define the optimizer for the phase0. We use AdamW with a weight decay of 1e-5. 
    # For bias, we don't use weight decay, and for head_w and head_cross, neither,
    # since these weights are already regularized by lambda_smooth, therefore, 
    # it would be double regularization.
    # Let us see it:
    # AdamW: L_total = L_task + \lambda · ||w||^2
    # Smooth: L_smooth = \lambda_smooth · mean ( (w · ddBx)^2 + ... )
    # Total: L_lotal = L_huber + L_positivity + \lambda_smooth · mean ( (w · ddBx)^2 + ... ) + \lambda · ||w||^2
    # We see then that the weight decay is applied twice, for smoothness and for the weights.
    decay, no_decay = [], []
    for name, p in m0.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    # Define AdamW phase0.
    opt_p0 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p0,
    )

    # Define the scheduler for the phase0. 
    # Mode = "min" means that the learning rate will be reduced when the validation loss
    # does not improve for PATIENCE epochs.
    # Factor = 0.5 means that the learning rate will be reduced by a factor of 0.5.
    # Patience = 10 means that the learning rate will be reduced after 10 epochs of no improvement.
    # Min_lr = 1e-5 means that the learning rate will not be reduced below 1e-5.
    sch_p0 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    # Run the training for the phase0.
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE, 
                 ckpt_p0, device, neighbor_meta, "P0")

    # ── Phase 1: Unlock spline weights with smoothed targets ───────────────────
    # Building on the stable beta and b from Phase 0, this phase introduces the
    # spline flexibility that was previously frozen:
    #
    #   1. The model is initialized from the Phase 0 checkpoint. The spline
    #      weights (head_w) are unfrozen (requires_grad=True), allowing the
    #      model to learn non-linear price responses beyond the log-linear baseline.
    #
    #   2. Training uses the non-smoothed data (train_loader / val_loader),
    #      unlike Phase 0 which trained on rolling-average targets.
    #
    # By the end of this phase, the spline shapes are well fit to the raw demand
    # signal.

    # Build the model with first-order and second-order cross-price effects 
    # and enforcing negative beta.
    m1 = make_model(enforce_negative_beta=True, use_cross=True)
    # Load the state dict from the Phase 0 checkpoint.
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    # Unfreeze the nonlinear parameters.
    unfreeze_nonlinear(m1)

    # We define the loss function for the phase1. Pure fit to the training data.
    loss_p1 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth,
        lambda_elast=lambda_elast,
        reduction="mean"
    )
    # The same as before. Avoiding double regularization.
    decay, no_decay = [], []
    for name, p in m1.named_parameters():
        if not p.requires_grad:
            continue
        if (("head_w" in name) or ("head_cross" in name)
                or ("head_beta_cross" in name) or name.endswith("bias")):
            no_decay.append(p)
        else:
            decay.append(p)

    # Define AdamW phase1. As before.
    opt_p1 = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 1e-5},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr_p1,
    )

    # Define the scheduler for the phase1. As before.
    sch_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5
    )

    # Run the training for the phase1.
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE, 
                 ckpt_p1, device, neighbor_meta, "P1")

    # Load the state dict from the Phase 1 checkpoint.
    m1.load_state_dict(torch.load(ckpt_p1, map_location=device))

    # Freeze P* once on the converged model: compute the global mean score matrix
    # over the full training set, then fix the sparse neighbor graph.
    # From this point on, run() uses the O(B * E * d_attn) sparse path.
    m1.eval()
    selector = m1.head.neighbor_selector
    def h_iter(loader):
        with torch.no_grad():
            for batch in loader:
                batch  = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
                tokens = m1.context_builder(batch)   # (B, n, d_token)
                h      = m1.head.encoder(tokens)     # (B, n, d_hidden)
                yield h
    global_mean = selector.accumulate_mean_scores(
        h_iter(train_loader),
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )
    selector.freeze_graph(
        global_mean,
        category=neighbor_meta["category"],
        brand=neighbor_meta["brand"],
        style=neighbor_meta["style"],
        liters=neighbor_meta["liters"],
    )

    # Compute the prediction metrics for the global metrics and the elasticity score.
    pred_metrics = compute_global_metrics(m1, val_loader, device)
    elast_metrics = compute_elasticity_score(m1, val_loader, device)

    out = {
        "trial_id": trial_id,
        "fold": fold_id,
        "seed": seed,
        "n_train": len(train_wide),
        "n_val": len(val_wide),
        **pred_metrics,
        **elast_metrics,
    }

    print(
        f"trial={trial_id} fold={fold_id} seed={seed} | "
        f"R2={out['r2_val']:.4f} MAE={out['mae_val']:.4f} | "
        f"ElastScore={out['elast_score']:.4f} | "
        f"own[pct={100*out['own_in_range']:.1f}% med={out['own_elasticity_median']:.2f}] "
        f"cross[pct={100*out['cross_in_range']:.1f}% med={out['cross_elasticity_median']:.2f}]"
    )
    # Remove the checkpoint files. For each trial, we have 2 checkpoints.
    # If we don't remove them, the folder will be full of checkpoints and our
    # hard drive will run out of space.
    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return out

print("build_and_train redefinided")

build_and_train redefinided


In [13]:
# Objective function to optimize in the hyperparameter search (Optuna).
trial_records = []
def objective(trial):
    # Define the parameters to optimize.
    params = {
        "N_KNOTS":             trial.suggest_int("N_KNOTS", 2, 16),
        "HIDDEN_KEY":          trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())),
        "DROPOUT":             trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":               trial.suggest_float("LR_P0", 1e-4, 1e-2, log=True),
        "LR_P1":               trial.suggest_float("LR_P1", 1e-5, 5e-3, log=True),
        "LAMBDA_SMOOTH": trial.suggest_float("LAMBDA_SMOOTH", 1e-5, 0.2, log=True),
        "LAMBDA_ELAST":  trial.suggest_float("LAMBDA_ELAST",  1e-5, 0.2, log=True),
        "BATCH_SIZE":          trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
    }

    print(f"\n{'='*70}")
    print(f"Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*70}")

    # Run the training for each fold and seed.
    run_rows = []
    for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
        for seed in TUNE_SEEDS:
            row = build_and_train(
                params=params,
                train_fold=train_fold,
                val_fold=val_fold,
                fold_id=fold_id,
                seed=seed,
                trial_id=trial.number,
            )
            run_rows.append(row)

    # Create a DataFrame from the run_rows.
    df_trial = pd.DataFrame(run_rows)

    # Compute the mean and standard deviation of the R2.
    mean_r2 = float(df_trial["r2_val"].mean())
    std_r2  = float(df_trial["r2_val"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    # Compute the mean and standard deviation of the Elasticity Score.
    mean_elast = float(df_trial["elast_score"].mean())
    std_elast  = float(df_trial["elast_score"].std(ddof=1)) if len(df_trial) > 1 else 0.0

    # Compute the mean and standard deviation of the MAE.
    mean_mae  = float(df_trial["mae_val"].mean())
    mean_rmse = float(df_trial["rmse_val"].mean())

    # Compute the robust score. We try to penalize the variance between folds
    # and rewards those trials that are more stable across folds. We set 0.25 
    # to control how much we penalize the variance.
    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    # Set the user attributes for the trial.
    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("mean_mae", mean_mae)
    trial.set_user_attr("mean_rmse", mean_rmse)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("robust_elast", robust_elast)

    # We build the historical records for the trials because, at the end,
    # we want to analyze the performance of the trials.
    df_trial["trial"] = trial.number
    for k, v in params.items():
        df_trial[k] = v
    trial_records.extend(df_trial.to_dict(orient="records"))

    print(
        f"Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast_Score={mean_elast:.4f} std_Elast_Score={std_elast:.4f} "
        f"robust_Elast_Score={robust_elast:.4f}"
    )

    return robust_r2, robust_elast

# Optuna Study

In [14]:
# We set the study name and the storage path.
study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="hparam_pareto_kfold_seed",
    storage="sqlite:///../results/hparam_pareto_kfold_seed.db",
    load_if_exists=True,
)

# We optimize the objective function.
# BE CAREFUL: This can take a while! 1 trial can take 15 min for a GPU - RTX5070Ti 
study.optimize(objective, n_trials=100)

print(f"\nTrials completed: {len(study.trials)}")

[I 2026-05-08 20:06:50,425] A new study created in RDB with name: hparam_pareto_kfold_seed



Trial 0
  N_KNOTS: 10
  HIDDEN_KEY: 64_32
  DROPOUT: 0.07538628034419735
  LR_P0: 0.00825609776924115
  LR_P1: 1.6977665522414735e-05
  LAMBDA_SMOOTH: 0.1563683390103826
  LAMBDA_ELAST: 0.03694696247671341
  BATCH_SIZE: 512
trial=0 fold=0 seed=11 | R2=0.5720 MAE=0.6560 | ElastScore=0.8780 | own[pct=100.0% med=-1.51] cross[pct=82.0% med=0.46]
trial=0 fold=0 seed=29 | R2=0.5131 MAE=0.7126 | ElastScore=0.8736 | own[pct=100.0% med=-1.47] cross[pct=84.2% med=0.49]
trial=0 fold=0 seed=42 | R2=0.5760 MAE=0.6526 | ElastScore=0.7451 | own[pct=100.0% med=-1.05] cross[pct=90.7% med=0.21]
trial=0 fold=1 seed=11 | R2=0.4247 MAE=0.6569 | ElastScore=0.6900 | own[pct=100.0% med=-0.95] cross[pct=83.7% med=0.20]
trial=0 fold=1 seed=29 | R2=0.3708 MAE=0.6942 | ElastScore=0.5961 | own[pct=100.0% med=-0.71] cross[pct=81.1% med=0.43]
trial=0 fold=1 seed=42 | R2=0.3744 MAE=0.6851 | ElastScore=0.6363 | own[pct=100.0% med=-0.78] cross[pct=85.6% med=0.23]
trial=0 fold=2 seed=11 | R2=0.1014 MAE=0.6578 | ElastSc

[I 2026-05-08 20:23:19,205] Trial 0 finished with values: [0.26580632876611987, 0.6899484640409532] and parameters: {'N_KNOTS': 10, 'HIDDEN_KEY': '64_32', 'DROPOUT': 0.07538628034419735, 'LR_P0': 0.00825609776924115, 'LR_P1': 1.6977665522414735e-05, 'LAMBDA_SMOOTH': 0.1563683390103826, 'LAMBDA_ELAST': 0.03694696247671341, 'BATCH_SIZE': 512}.


Trial 0 summary | mean_R2=0.3249 std_R2=0.2362 robust_R2=0.2658 | mean_Elast_Score=0.7187 std_Elast_Score=0.1149 robust_Elast_Score=0.6899

Trial 1
  N_KNOTS: 4
  HIDDEN_KEY: 128_64
  DROPOUT: 0.06976421282464741
  LR_P0: 0.0009837565386734575
  LR_P1: 0.0002664167990262848
  LAMBDA_SMOOTH: 4.15287346404372e-05
  LAMBDA_ELAST: 0.009900092278151786
  BATCH_SIZE: 1024
trial=1 fold=0 seed=11 | R2=0.7367 MAE=0.5127 | ElastScore=0.7432 | own[pct=72.6% med=-1.57] cross[pct=89.2% med=0.60]
trial=1 fold=0 seed=29 | R2=0.7334 MAE=0.5150 | ElastScore=0.7393 | own[pct=77.7% med=-1.36] cross[pct=95.6% med=0.59]
trial=1 fold=0 seed=42 | R2=0.7390 MAE=0.5065 | ElastScore=0.8270 | own[pct=77.7% med=-1.70] cross[pct=94.5% med=0.57]
trial=1 fold=1 seed=11 | R2=0.7224 MAE=0.4589 | ElastScore=0.6842 | own[pct=82.6% med=-1.28] cross[pct=76.2% med=0.54]
trial=1 fold=1 seed=29 | R2=0.7095 MAE=0.4652 | ElastScore=0.6606 | own[pct=87.7% med=-1.06] cross[pct=80.9% med=0.57]
trial=1 fold=1 seed=42 | R2=0.7206 M

[I 2026-05-08 20:33:24,686] Trial 1 finished with values: [0.641915511128989, 0.7674719017029311] and parameters: {'N_KNOTS': 4, 'HIDDEN_KEY': '128_64', 'DROPOUT': 0.06976421282464741, 'LR_P0': 0.0009837565386734575, 'LR_P1': 0.0002664167990262848, 'LAMBDA_SMOOTH': 4.15287346404372e-05, 'LAMBDA_ELAST': 0.009900092278151786, 'BATCH_SIZE': 1024}.


...

Trial 97 summary | mean_R2=0.6695 std_R2=0.0960 robust_R2=0.6455 | mean_Elast_Score=0.8614 std_Elast_Score=0.0693 robust_Elast_Score=0.8441

Trial 98
  N_KNOTS: 10
  HIDDEN_KEY: 192_96
  DROPOUT: 0.1825181580059273
  LR_P0: 0.0006923141514014833
  LR_P1: 0.003497386771216078
  LAMBDA_SMOOTH: 7.080792225748593e-05
  LAMBDA_ELAST: 0.001205186194362949
  BATCH_SIZE: 512
trial=98 fold=0 seed=11 | R2=0.7339 MAE=0.5122 | ElastScore=0.6311 | own[pct=71.1% med=-1.30] cross[pct=77.5% med=0.60]
trial=98 fold=0 seed=29 | R2=0.7351 MAE=0.5068 | ElastScore=0.5607 | own[pct=71.3% med=-0.98] cross[pct=80.1% med=0.60]
trial=98 fold=0 seed=42 | R2=0.7238 MAE=0.5134 | ElastScore=0.5941 | own[pct=70.4% med=-1.15] cross[pct=78.6% med=0.59]
trial=98 fold=1 seed=11 | R2=0.7042 MAE=0.4703 | ElastScore=0.6766 | own[pct=87.2% med=-1.22] cross[pct=70.5% med=0.59]
trial=98 fold=1 seed=29 | R2=0.7104 MAE=0.4719 | ElastScore=0.6747 | own[pct=87.4% med=-1.24] cross[pct=68.4% med=0.57]
trial=98 fold=1 seed=42 | R2=0

[I 2026-05-09 23:32:45,668] Trial 98 finished with values: [0.6410490803759735, 0.6978303035981177] and parameters: {'N_KNOTS': 10, 'HIDDEN_KEY': '192_96', 'DROPOUT': 0.1825181580059273, 'LR_P0': 0.0006923141514014833, 'LR_P1': 0.003497386771216078, 'LAMBDA_SMOOTH': 7.080792225748593e-05, 'LAMBDA_ELAST': 0.001205186194362949, 'BATCH_SIZE': 512}.


Trial 98 summary | mean_R2=0.6629 std_R2=0.0874 robust_R2=0.6410 | mean_Elast_Score=0.7314 std_Elast_Score=0.1341 robust_Elast_Score=0.6978

Trial 99
  N_KNOTS: 8
  HIDDEN_KEY: 64_32
  DROPOUT: 0.07538628034419735
  LR_P0: 0.002348396895198714
  LR_P1: 0.00026217204396461875
  LAMBDA_SMOOTH: 0.0631257921174423
  LAMBDA_ELAST: 0.0019420567494715729
  BATCH_SIZE: 512
trial=99 fold=0 seed=11 | R2=0.7121 MAE=0.5326 | ElastScore=0.7161 | own[pct=100.0% med=-1.04] cross[pct=82.5% med=0.32]
trial=99 fold=0 seed=29 | R2=0.6930 MAE=0.5526 | ElastScore=0.7219 | own[pct=100.0% med=-1.04] cross[pct=84.8% med=0.34]
trial=99 fold=0 seed=42 | R2=0.7114 MAE=0.5360 | ElastScore=0.8412 | own[pct=100.0% med=-1.42] cross[pct=79.2% med=0.32]
trial=99 fold=1 seed=11 | R2=0.6695 MAE=0.4933 | ElastScore=0.8469 | own[pct=100.0% med=-1.42] cross[pct=81.7% med=0.49]
trial=99 fold=1 seed=29 | R2=0.6874 MAE=0.4793 | ElastScore=0.8795 | own[pct=100.0% med=-1.50] cross[pct=83.0% med=0.34]
trial=99 fold=1 seed=42 | R

[I 2026-05-09 23:48:55,560] Trial 99 finished with values: [0.6022469098570402, 0.8105999569508163] and parameters: {'N_KNOTS': 8, 'HIDDEN_KEY': '64_32', 'DROPOUT': 0.07538628034419735, 'LR_P0': 0.002348396895198714, 'LR_P1': 0.00026217204396461875, 'LAMBDA_SMOOTH': 0.0631257921174423, 'LAMBDA_ELAST': 0.0019420567494715729, 'BATCH_SIZE': 512}.


Trial 99 summary | mean_R2=0.6270 std_R2=0.0989 robust_R2=0.6022 | mean_Elast_Score=0.8281 std_Elast_Score=0.0699 robust_Elast_Score=0.8106

Trials completed: 100


# Summary

In [17]:
# We create a DataFrame with the summary of the trials.
summary_rows = []
for t in study.trials:
    if t.values is None:
        continue
    # We build the row for the summary.
    row = {
        "trial": t.number,
        "mean_r2": t.user_attrs.get("mean_r2", np.nan),
        "std_r2": t.user_attrs.get("std_r2", np.nan),
        "robust_r2": t.user_attrs.get("robust_r2", np.nan),     
        "robust_elast": t.user_attrs.get("robust_elast", np.nan), 
        "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
        "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
        "mean_mae": t.user_attrs.get("mean_mae", np.nan),
        "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
        **t.params,
    }
    summary_rows.append(row)

# We sort the trials by the mean R2 and Elasticity Score.
df_trials_summary = pd.DataFrame(summary_rows).sort_values(
    ["mean_r2", "mean_elast_score"], ascending=[False, False]
)
# The first 15 trials are printed.
print(df_trials_summary.head(15).to_string(index=False))

 trial  mean_r2   std_r2  robust_r2  robust_elast  mean_elast_score  std_elast_score  mean_mae  mean_rmse  N_KNOTS HIDDEN_KEY  DROPOUT    LR_P0    LR_P1  LAMBDA_SMOOTH  LAMBDA_ELAST  BATCH_SIZE
    53 0.678860 0.099782   0.653915      0.891796          0.907037         0.060963  0.461505   0.599352        2      64_32 0.276040 0.001203 0.001083       0.000058      0.056729         256
    48 0.669829 0.097842   0.645368      0.899056          0.905619         0.026254  0.472361   0.608769        7      64_32 0.286285 0.000352 0.000831       0.003613      0.000767         256
    87 0.669521 0.097643   0.645111      0.854091          0.874055         0.079857  0.472259   0.609225       11      64_32 0.286285 0.000352 0.000831       0.000362      0.030960         512
    97 0.669494 0.096011   0.645491      0.844126          0.861445         0.069279  0.471673   0.609726       10      64_32 0.182518 0.000330 0.001150       0.001140      0.002227         256
    33 0.669409 0.092776   0.6

# Best Trial

In [19]:
# We set the robust score. 
df_trials_summary["robust_score"] = (
    df_trials_summary["robust_r2"].fillna(0.0)
    +  df_trials_summary["robust_elast"].fillna(0.0) 
)

# IMPORTANT! Don't confuse with the robust_r2 and robust_elast. Here,
# we are using the robust_score to select the best trial. An unique value
# for the selection of the best trial..

# We select the best trial.
best_row = df_trials_summary.sort_values("robust_score", ascending=False).iloc[0]
# We create the payload for the best trial.
best_trial_payload = {
    "trial": int(best_row["trial"]),
    "robust_score": float(best_row["robust_score"]),
    "mean_r2": float(best_row["mean_r2"]),
    "std_r2": float(best_row["std_r2"]),
    "mean_elast_score": float(best_row["mean_elast_score"]),
    "std_elast_score": float(best_row["std_elast_score"]),
    "params": {
        "N_KNOTS":            int(best_row["N_KNOTS"]),
        "HIDDEN_KEY":         str(best_row["HIDDEN_KEY"]),
        "DROPOUT":            float(best_row["DROPOUT"]),
        "LR_P0":              float(best_row["LR_P0"]),
        "LR_P1":              float(best_row["LR_P1"]),
        "LAMBDA_SMOOTH":      float(best_row["LAMBDA_SMOOTH"]),
        "LAMBDA_ELAST":       float(best_row["LAMBDA_ELAST"]),
        "BATCH_SIZE":         int(best_row["BATCH_SIZE"]),
    }
}

# We save the best trial.
with open(BEST_TRIAL_PATH, "w", encoding="utf-8") as f:
    json.dump(best_trial_payload, f, indent=2, ensure_ascii=False)

# We save the summary of the trials.
df_trials_summary.to_csv(TRIAL_SUMMARY_PATH, index=False)

print("Best trial saved in:", BEST_TRIAL_PATH)
print("Trials summary saved in:", TRIAL_SUMMARY_PATH)
print(json.dumps(best_trial_payload, indent=2, ensure_ascii=False))

Best trial saved in: ../results/best_trial_params.json
Trials summary saved in: ../results/nn_hparam_trials_summary.csv
{
  "trial": 28,
  "robust_score": 1.582599359822807,
  "mean_r2": 0.6635632657400977,
  "std_r2": 0.10203748042318032,
  "mean_elast_score": 0.950533724327848,
  "std_elast_score": 0.023953040557374308,
  "params": {
    "N_KNOTS": 3,
    "HIDDEN_KEY": "256_128_64",
    "DROPOUT": 0.2546804232627202,
    "LR_P0": 0.0016856369391482482,
    "LR_P1": 0.001624635483799384,
    "LAMBDA_SMOOTH": 0.035143815193994093,
    "LAMBDA_ELAST": 0.04449893284701117,
    "BATCH_SIZE": 256
  }
}
